# LeetCode 3671 — Sum of Beautiful Subsequences bằng Python

- Link: https://leetcode.com/problems/sum-of-beautiful-subsequences/
- Độ khó: **Hard**
- Chủ đề: Đếm **dãy con tăng ngặt** bằng Fenwick Tree, số học trên ước (**bao hàm – loại trừ theo bội**, **hàm phi Euler**), sàng ước

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def totalBeauty(self, nums: List[int]) -> int:
```

## 1. Mục tiêu học

1. Viết lại đáp án thành `Σ_{dãy con tăng ngặt S} gcd(S)`.
2. Đếm `g(d)` = số dãy con tăng ngặt mà **mọi phần tử chia hết cho `d`** — chỉ cần lọc các phần tử là bội của `d` rồi đếm dãy con tăng bằng Fenwick.
3. Từ `g(d)` suy ra số dãy có gcd **đúng bằng** `d`: `f(d) = g(d) − Σ_{k≥2} f(kd)`.
4. Mẹo gọn hơn: `gcd(S) = Σ_{d | gcd(S)} φ(d)` ⇒ đáp án `= Σ_d φ(d)·g(d)` — không cần tính `f`.
5. Kiểm chứng bằng vét cạn.

## 2. Đề bài & ví dụ

Độ đẹp của `g` = `g ×` (số dãy con **tăng ngặt** có gcd đúng bằng `g`). Tính tổng độ đẹp với mọi `g`, modulo `10^9 + 7`.

**Ví dụ 1:** `[1,2,3]` → `10`.

**Ví dụ 2:** `[4,6]` → `12`.

**Ràng buộc:** `n ≤ 10^4`, `nums[i] ≤ 7·10^4`.

In [1]:
from typing import List
from math import gcd
from functools import reduce
from collections import defaultdict
import random, time

MOD = 10**9 + 7
examples = [([1, 2, 3], 10), ([4, 6], 12)]
for a, exp in examples:
    print(a, "-> kỳ vọng", exp)

[1, 2, 3] -> kỳ vọng 10
[4, 6] -> kỳ vọng 12


## 3. Kiến thức nền

### 3.1. Đếm dãy con tăng ngặt bằng Fenwick

Duyệt từ trái sang phải; số dãy con tăng ngặt kết thúc tại `x` là `1 + Σ_{y < x, y đứng trước} cnt(y)`. Fenwick theo **giá trị** (đã nén) cho tổng tiền tố "giá trị `< x`" trong `O(log)`.

In [2]:
def count_increasing(arr):
    vals = sorted(set(arr)); rank = {v: i + 1 for i, v in enumerate(vals)}; m = len(vals)
    bit = [0] * (m + 1); total = 0
    for x in arr:
        r = rank[x] - 1; s = 0
        while r > 0: s += bit[r]; r -= r & -r
        c = (1 + s) % MOD; total += c
        r = rank[x]
        while r <= m: bit[r] = (bit[r] + c) % MOD; r += r & -r
    return total % MOD

print(count_increasing([1, 2, 3]), count_increasing([3, 2, 1]), count_increasing([2, 2]))

7 3 2


### 3.2. Hàm phi Euler và đẳng thức `Σ_{d|n} φ(d) = n`

`φ(d)` = số số trong `1..d` nguyên tố cùng nhau với `d`. Đẳng thức: chia các phân số `1/n, 2/n, …, n/n` theo mẫu số tối giản `d | n` — có đúng `φ(d)` phân số mẫu `d`.

In [3]:
def phi_sieve(N):
    phi = list(range(N + 1))
    for p in range(2, N + 1):
        if phi[p] == p:
            for k in range(p, N + 1, p): phi[k] -= phi[k] // p
    return phi

PHI = phi_sieve(100)
print([PHI[d] for d in range(1, 13)], "| Σ_{d|12} φ(d) =", sum(PHI[d] for d in range(1, 13) if 12 % d == 0))

[1, 1, 2, 2, 4, 2, 6, 4, 6, 4, 10, 4] | Σ_{d|12} φ(d) = 12


### 3.3. Bao hàm – loại trừ theo bội

`g(d) = Σ_{k ≥ 1} f(kd)` (mỗi dãy có gcd là bội của `d` được đếm ở `g(d)`). Duyệt `d` từ lớn xuống: `f(d) = g(d) − Σ_{k≥2} f(kd)`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Đáp án `= Σ_S gcd(S)` trên mọi dãy con tăng ngặt `S`. Thay `gcd(S) = Σ_{d | gcd(S)} φ(d)` và đổi thứ tự tổng:

`Σ_S Σ_{d | gcd(S)} φ(d) = Σ_d φ(d) · #{S : d | gcd(S)} = Σ_d φ(d) · g(d)`.

Ví dụ 2 `[4,6]`: `g(1) = 3`, `g(2) = 3`, `g(3) = 1` (`[6]`), `g(4) = 1`, `g(6) = 1`. `Σ φ(d)g(d) = 1·3 + 1·3 + 2·1 + 2·1 + 2·1 = 12`. ✔

## 5. Cách 1 — Brute Force

In [4]:
def brute(nums):
    n = len(nums); total = 0
    for mask in range(1, 1 << n):
        s = [nums[i] for i in range(n) if mask >> i & 1]
        if all(s[i] < s[i + 1] for i in range(len(s) - 1)):
            total += reduce(gcd, s)
    return total % MOD

for a, exp in examples:
    print(brute(a), "kỳ vọng", exp)

10 kỳ vọng 10
12 kỳ vọng 12


**Nút thắt:** `2^n`.

## 6. Ý tưởng thuật toán / bước đột phá

1. Với mỗi `d`, danh sách `L_d` = các phần tử là bội của `d` theo thứ tự gốc. Xây bằng cách duyệt **ước** của từng phần tử (sàng ước tới `7·10^4`).
2. `g(d) = count_increasing(L_d)`.
3. Đáp án `Σ φ(d) g(d)`.

Tổng độ dài các `L_d` = `Σ_i τ(nums[i])` (số ước) — trung bình vài chục, tối đa ~100 mỗi số.

**Tối ưu nhỏ:** nếu mọi phần tử trong `L_d` bằng nhau (rất hay gặp với `d` lớn), mọi dãy con tăng ngặt chỉ có một phần tử ⇒ `g(d) = |L_d|`, bỏ qua Fenwick. Lời giải nộp dùng mẹo này. Ngoài ra chỉ sàng ước cho các giá trị **có mặt** trong `nums` để tiết kiệm bộ nhớ.

## 7. Các cách tiếp theo

- **Cách 2:** DP `O(n²)` theo gcd: `dp[i]` = từ điển `gcd → số dãy` kết thúc tại `i`.
- **Cách 3:** Fenwick theo từng `d` + bao hàm – loại trừ `f(d) = g(d) − Σ f(kd)` (theo doocs).
- **Cách 4:** Fenwick theo từng `d` + `Σ φ(d) g(d)` (lời giải nộp).

### 7.1. Cách 2 — DP theo gcd `O(n²)`

In [5]:
def dp_gcd(nums):
    n = len(nums); dp = [None] * n; total = 0
    for i in range(n):
        cur = defaultdict(int); cur[nums[i]] += 1
        for j in range(i):
            if nums[j] < nums[i]:
                for g, c in dp[j].items():
                    cur[gcd(g, nums[i])] += c
        dp[i] = cur
        total += sum(g * c for g, c in cur.items())
    return total % MOD

for a, exp in examples:
    print(dp_gcd(a), "kỳ vọng", exp)

10 kỳ vọng 10
12 kỳ vọng 12


### 7.2. Cách 3 — Fenwick theo `d` + bao hàm – loại trừ

In [6]:
def divisors_table(N):
    divs = [[] for _ in range(N + 1)]
    for d in range(1, N + 1):
        for k in range(d, N + 1, d): divs[k].append(d)
    return divs

def lists_by_divisor(nums):
    N = max(nums); divs = divisors_table(N); L = defaultdict(list)
    for x in nums:
        for d in divs[x]: L[d].append(x)
    return N, L

def fenwick_inclusion_exclusion(nums):
    N, L = lists_by_divisor(nums)
    g = [0] * (N + 1)
    for d, arr in L.items(): g[d] = count_increasing(arr)
    f = [0] * (N + 1)
    for d in range(N, 0, -1):
        f[d] = g[d]
        for k in range(2 * d, N + 1, d): f[d] -= f[k]
        f[d] %= MOD
    return sum(d * f[d] for d in range(1, N + 1)) % MOD

for a, exp in examples:
    print(fenwick_inclusion_exclusion(a), "kỳ vọng", exp)

10 kỳ vọng 10
12 kỳ vọng 12


### 7.3. Cách 4 — Fenwick theo `d` + hàm phi (lời giải chính)

In [7]:
def fenwick_phi(nums):
    N, L = lists_by_divisor(nums)
    phi = phi_sieve(N)
    return sum(phi[d] * count_increasing(arr) for d, arr in L.items()) % MOD

for a, exp in examples:
    print(fenwick_phi(a), "kỳ vọng", exp)

10 kỳ vọng 10
12 kỳ vọng 12


## 8. TRACE từng bước

In [8]:
def trace(nums):
    N, L = lists_by_divisor(nums); phi = phi_sieve(N); total = 0
    for d in sorted(L):
        gd = count_increasing(L[d]); total += phi[d] * gd
        print(f"d={d:>2}: bội của d = {L[d]}, g(d) = {gd}, φ(d) = {phi[d]}, đóng góp {phi[d] * gd} -> tổng {total}")
    print("đáp án:", total % MOD, "| brute:", brute(nums))

trace([4, 6])

d= 1: bội của d = [4, 6], g(d) = 3, φ(d) = 1, đóng góp 3 -> tổng 3
d= 2: bội của d = [4, 6], g(d) = 3, φ(d) = 1, đóng góp 3 -> tổng 6
d= 3: bội của d = [6], g(d) = 1, φ(d) = 2, đóng góp 2 -> tổng 8
d= 4: bội của d = [4], g(d) = 1, φ(d) = 2, đóng góp 2 -> tổng 10
d= 6: bội của d = [6], g(d) = 1, φ(d) = 2, đóng góp 2 -> tổng 12
đáp án: 12 | brute: 12


In [9]:
trace([2, 6, 4, 12, 3])

d= 1: bội của d = [2, 6, 4, 12, 3], g(d) = 13, φ(d) = 1, đóng góp 13 -> tổng 13
d= 2: bội của d = [2, 6, 4, 12], g(d) = 11, φ(d) = 1, đóng góp 11 -> tổng 24
d= 3: bội của d = [6, 12, 3], g(d) = 4, φ(d) = 2, đóng góp 8 -> tổng 32
d= 4: bội của d = [4, 12], g(d) = 3, φ(d) = 2, đóng góp 6 -> tổng 38
d= 6: bội của d = [6, 12], g(d) = 3, φ(d) = 2, đóng góp 6 -> tổng 44
d=12: bội của d = [12], g(d) = 1, φ(d) = 4, đóng góp 4 -> tổng 48
đáp án: 48 | brute: 48


## 9. Code nộp LeetCode

In [10]:
class Solution:
    def totalBeauty(self, nums: List[int]) -> int:
        MOD = 10**9 + 7
        N = max(nums)
        phi = list(range(N + 1))
        for p in range(2, N + 1):
            if phi[p] == p:
                for k in range(p, N + 1, p):
                    phi[k] -= phi[k] // p
        present = [False] * (N + 1)
        for x in nums:
            present[x] = True
        groups = defaultdict(list)                     # d -> danh sách bội theo thứ tự gốc
        divs = [[] for _ in range(N + 1)]
        for d in range(1, N + 1):
            for k in range(d, N + 1, d):
                if present[k]:
                    divs[k].append(d)
        for x in nums:
            for d in divs[x]:
                groups[d].append(x)
        ans = 0
        for d, arr in groups.items():
            vals = sorted(set(arr))
            if len(vals) == 1:                         # mọi phần tử bằng nhau: chỉ dãy một phần tử
                ans += phi[d] * len(arr)
                continue
            rank = {v: i + 1 for i, v in enumerate(vals)}
            m = len(vals)
            bit = [0] * (m + 1)
            tot = 0
            for x in arr:
                r = rank[x] - 1
                s = 0
                while r > 0:
                    s += bit[r]
                    r -= r & -r
                c = (1 + s) % MOD
                tot += c
                r = rank[x]
                while r <= m:
                    bit[r] = (bit[r] + c) % MOD
                    r += r & -r
            ans += phi[d] * (tot % MOD)
        return ans % MOD

sol = Solution()
for a, exp in examples:
    got = sol.totalBeauty(a)
    print(a, "->", got, "✔" if got == exp else "✘")

[1, 2, 3] -> 10 ✔
[4, 6] -> 12 ✔


## 10. Vì sao thuật toán đúng

1. **Tổng độ đẹp** `= Σ_g g · #{S : gcd(S) = g} = Σ_S gcd(S)` (gom theo gcd).
2. **`g(d)` đúng:** `d | gcd(S)` ⇔ mọi phần tử của `S` chia hết cho `d` ⇔ `S` là dãy con tăng ngặt của `L_d` (giữ thứ tự gốc). Fenwick đếm đúng số dãy con tăng ngặt (3.1): mỗi dãy được đếm tại phần tử cuối của nó.
3. **Phi Euler:** `gcd(S) = Σ_{d|gcd(S)} φ(d)` (3.2) ⇒ đổi thứ tự tổng ⇒ `Σ_d φ(d) g(d)`.
4. **Bao hàm – loại trừ (Cách 3):** `g(d) = Σ_k f(kd)` ⇒ giải ngược từ `d` lớn xuống; rồi `Σ d f(d)`. Hai cách tương đương vì `Σ_d φ(d) Σ_{k} f(kd) = Σ_G f(G) Σ_{d|G} φ(d) = Σ_G G f(G)`.

## 11. Phân tích Time / Space Complexity

Với `V = max(nums)`, `T = Σ_i τ(nums[i])` (tổng số ước):

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(2^n · n)` | `O(n)` |
| 2. DP gcd | `O(n² · #gcd)` | `O(n · #gcd)` |
| 3. Fenwick + bao hàm | `O(V log V + T log n + V log V)` | `O(V + T)` |
| 4. Fenwick + φ | `O(V log V + T log n)` | `O(V + T)` |

Trường hợp xấu (mọi số có rất nhiều ước, `T ≈ 10^6`) Python mất vài giây; dữ liệu ngẫu nhiên nhanh hơn nhiều.

## 12. Bộ test

In [11]:
METHODS = [brute, dp_gcd, fenwick_inclusion_exclusion, fenwick_phi, lambda a: Solution().totalBeauty(a)]
for a, exp in examples:
    assert all(f(a) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [12]:
# 12.2 ca biên
assert Solution().totalBeauty([7]) == 7
assert Solution().totalBeauty([5, 5, 5]) == 15                 # chỉ các dãy một phần tử
assert Solution().totalBeauty([3, 2, 1]) == 6
print("Ca biên: OK")

Ca biên: OK


In [13]:
# 12.3 ngẫu nhiên
rng = random.Random(3671)
for _ in range(800):
    a = [rng.randint(1, 30) for _ in range(rng.randint(1, 11))]
    b = brute(a)
    assert all(f(a) == b for f in METHODS[1:]), a
print("800 test ngẫu nhiên: OK")

800 test ngẫu nhiên: OK


In [14]:
# 12.4 test lớn
rng = random.Random(1)
a = [rng.randint(1, 7 * 10**4) for _ in range(10**4)]
t0 = time.time(); x = Solution().totalBeauty(a); t1 = time.time()
print("ngẫu nhiên: đáp án", x, "| thời gian", round(t1 - t0, 2), "s | khớp Cách 3:", x == fenwick_inclusion_exclusion(a))
b = [rng.choice([55440, 50400, 45360, 27720]) for _ in range(10**4)]           # nhiều ước
t0 = time.time(); y = Solution().totalBeauty(b); t1 = time.time()
print("nhiều ước: đáp án", y, "| thời gian", round(t1 - t0, 2), "s")

ngẫu nhiên: đáp án 526653316 | thời gian 0.25 s | khớp Cách 3: True


nhiều ước: đáp án 938608839 | thời gian 0.43 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm dãy con **không giảm** (cho phép bằng nhau)

In [15]:
def wrong_nondecreasing(nums):
    n = len(nums); t = 0
    for mask in range(1, 1 << n):
        s = [nums[i] for i in range(n) if mask >> i & 1]
        if all(s[i] <= s[i + 1] for i in range(len(s) - 1)): t += reduce(gcd, s)      # SAI: <=
    return t

print("sai:", wrong_nondecreasing([2, 2]), "| đúng:", brute([2, 2]))

sai: 6 | đúng: 4


### Lỗi 2 — Cộng `d · g(d)` (quên khử phần đếm trùng)

In [16]:
def wrong_no_inclusion(nums):
    N, L = lists_by_divisor(nums)
    return sum(d * count_increasing(arr) for d, arr in L.items()) % MOD         # SAI

print("sai:", wrong_no_inclusion([4, 6]), "| đúng:", brute([4, 6]))

sai: 22 | đúng: 12


### Lỗi 3 — Fenwick dùng `≤` (truy vấn `prefix(rank)` thay vì `prefix(rank − 1)`)

In [17]:
def wrong_prefix(arr):
    vals = sorted(set(arr)); rank = {v: i + 1 for i, v in enumerate(vals)}; m = len(vals)
    bit = [0] * (m + 1); total = 0
    for x in arr:
        r = rank[x]; s = 0                                                         # SAI: gồm cả = x
        while r > 0: s += bit[r]; r -= r & -r
        c = 1 + s; total += c; r = rank[x]
        while r <= m: bit[r] += c; r += r & -r
    return total

print("sai:", wrong_prefix([3, 3]), "| đúng:", count_increasing([3, 3]))

sai: 3 | đúng: 2


### Lỗi 4 — Lọc bội của `d` nhưng **sắp xếp** lại danh sách

Dãy con phải giữ **thứ tự gốc**; sắp xếp làm mọi tổ hợp trở thành tăng.

In [18]:
print("giữ thứ tự:", count_increasing([6, 4, 2]), "| sắp xếp nhầm:", count_increasing(sorted([6, 4, 2])))

giữ thứ tự: 3 | sắp xếp nhầm: 7


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tổng gcd trên mọi đối tượng" ⇒ `Σ_d φ(d) · #{đối tượng mà d | gcd}`.
- "Số đối tượng có gcd đúng bằng `d`" ⇒ bao hàm – loại trừ theo bội, duyệt `d` giảm dần.
- "Đếm dãy con tăng ngặt" ⇒ Fenwick theo giá trị, truy vấn `prefix(rank − 1)`.
- Tổng số ước của các số `≤ 7·10^4` nhỏ ⇒ có thể xử lý "theo từng ước".
- Ba hàm số học hay gặp: `Σ_{d|n} φ(d) = n` (tổng gcd), `Σ_{d|n} μ(d) = [n = 1]` (đếm gcd = 1), `Σ_{d|n} 1 = τ(n)` (số ước) — nhớ chúng để "đổi thứ tự tổng" nhanh.

## 15. Biến thể của bài

1. **Dãy con bất kỳ** (không cần tăng): `g(d) = 2^{|L_d|} − 1`.
2. **Tổng lcm** thay vì gcd: khó hơn nhiều (lcm không có đẳng thức kiểu φ đơn giản trên bội).
3. **Mảng con liên tiếp** thay vì dãy con: gcd của mảng con kết thúc tại `i` có `O(log V)` giá trị khác nhau.
4. **Chỉ đếm dãy con tăng ngặt có gcd = 1**: `Σ μ(d) g(d)`.

In [19]:
# Biến thể 1 + 4
def sum_gcd_all_subsequences(nums):
    N, L = lists_by_divisor(nums); phi = phi_sieve(N)
    return sum(phi[d] * (pow(2, len(arr), MOD) - 1) for d, arr in L.items()) % MOD

def mobius(N):
    mu = [1] * (N + 1); mu[0] = 0; is_c = [False] * (N + 1); pr = []
    for i in range(2, N + 1):
        if not is_c[i]: pr.append(i); mu[i] = -1
        for p in pr:
            if i * p > N: break
            is_c[i * p] = True
            if i % p == 0: mu[i * p] = 0; break
            mu[i * p] = -mu[i]
    return mu

def count_coprime_increasing(nums):
    N, L = lists_by_divisor(nums); mu = mobius(N)
    return sum(mu[d] * count_increasing(arr) for d, arr in L.items()) % MOD

rng = random.Random(15)
for _ in range(300):
    a = [rng.randint(1, 20) for _ in range(rng.randint(1, 9))]; n = len(a)
    subs = [[a[i] for i in range(n) if m >> i & 1] for m in range(1, 1 << n)]
    assert sum_gcd_all_subsequences(a) == sum(reduce(gcd, s) for s in subs) % MOD
    inc = [s for s in subs if all(s[i] < s[i + 1] for i in range(len(s) - 1))]
    assert count_coprime_increasing(a) == sum(1 for s in inc if reduce(gcd, s) == 1) % MOD
print("Biến thể 1, 4 OK")

Biến thể 1, 4 OK


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. Vét cạn | Mọi dãy con | `O(2^n n)` | Kiểm chứng |
| 2. DP gcd | Từ điển gcd mỗi vị trí | `O(n² · #gcd)` | `n` nhỏ |
| 3. Fenwick + bao hàm | `f(d) = g(d) − Σ f(kd)` | `O(V log V + T log n)` | Khi cần phân phối gcd |
| 4. Fenwick + φ | `Σ φ(d) g(d)` | `O(V log V + T log n)` | **Lời giải chuẩn** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `totalBeauty([2, 4, 8])`.

**Bài 2 (tính tay).** Tính `g(d)` cho mọi `d` với `[6, 3, 9]` rồi suy ra đáp án.

**Bài 3 (điền TODO).** Viết `distribution(nums)` trả về từ điển `gcd → số dãy con tăng ngặt có gcd đó`.

**Bài 4 (tự viết lại).** Viết lại `count_increasing` bằng `bisect` + mảng tiền tố khi giá trị đã nén (không Fenwick) cho `n` nhỏ và so sánh.

**Bài 5 (mini challenge).** Tính tổng `gcd` trên mọi dãy con **giảm ngặt**.

In [20]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def distribution(nums):
    # TODO: f(d) từ Cách 3
    pass

In [21]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def total_beauty_decreasing(nums):
    # TODO: đảo dấu giá trị hoặc đảo mảng
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Dãy tăng: `[2],[4],[8],[2,4],[2,8],[4,8],[2,4,8]`. (2) gcd: `2,4,8,2,2,4,2`. (3) Tổng `24`.

**Bài 2.** (1) `L_1 = [6,3,9]`, `L_3 = [6,3,9]`, `L_6 = [6]`, `L_9 = [9]`, `L_2 = [6]`. (2) `g(1) = g(3) = 5` (`[6],[3],[9],[6,9],[3,9]`), `g(2) = g(6) = g(9) = 1`. (3) `φ(1)g(1) + φ(3)g(3) + φ(2)g(2) + φ(6)g(6) + φ(9)g(9) = 5 + 10 + 1 + 2 + 6 = 24`.

**Bài 3.** (1) Tính `g`. (2) `f` từ `d` lớn xuống. (3) Lọc `f(d) ≠ 0`.

**Bài 4.** (1) Duy trì `cnt[v]` theo giá trị. (2) Tổng các `cnt[u]` với `u < x`: duyệt tuyến tính. (3) `O(n²)` nhưng đơn giản.

**Bài 5.** (1) Giảm ngặt trên `nums` = tăng ngặt trên `[-x]`… nhưng gcd cần số dương. (2) Đảo **thứ tự mảng**: dãy giảm ngặt của `nums` = dãy tăng ngặt của `nums[::-1]`. (3) Gọi lại lời giải.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2
assert Solution().totalBeauty([2, 4, 8]) == 24 == brute([2, 4, 8])
assert Solution().totalBeauty([6, 3, 9]) == 24 == brute([6, 3, 9])
print("Bài 1, 2 OK:", Solution().totalBeauty([6, 3, 9]))

Bài 1, 2 OK: 24


In [23]:
# Bài 3, 4
def distribution_sol(nums):
    N, L = lists_by_divisor(nums)
    g = [0] * (N + 1)
    for d, arr in L.items(): g[d] = count_increasing(arr)
    f = [0] * (N + 1)
    for d in range(N, 0, -1):
        f[d] = (g[d] - sum(f[k] for k in range(2 * d, N + 1, d))) % MOD
    return {d: f[d] for d in range(1, N + 1) if f[d]}

def count_increasing_simple(arr):
    cnt = defaultdict(int); total = 0
    for x in arr:
        c = 1 + sum(v for u, v in cnt.items() if u < x)
        cnt[x] += c; total += c
    return total % MOD

rng = random.Random(4)
for _ in range(300):
    a = [rng.randint(1, 20) for _ in range(rng.randint(1, 9))]; n = len(a)
    inc = [s for s in ([a[i] for i in range(n) if m >> i & 1] for m in range(1, 1 << n)) if all(s[i] < s[i + 1] for i in range(len(s) - 1))]
    dist = defaultdict(int)
    for s in inc: dist[reduce(gcd, s)] += 1
    assert distribution_sol(a) == dict(dist)
    assert count_increasing_simple(a) == count_increasing(a)
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [24]:
# Bài 5
def total_beauty_decreasing_sol(nums):
    return Solution().totalBeauty(nums[::-1])

rng = random.Random(5)
for _ in range(300):
    a = [rng.randint(1, 20) for _ in range(rng.randint(1, 9))]; n = len(a)
    exp = sum(reduce(gcd, s) for s in ([a[i] for i in range(n) if m >> i & 1] for m in range(1, 1 << n))
              if all(s[i] > s[i + 1] for i in range(len(s) - 1))) % MOD
    assert total_beauty_decreasing_sol(a) == exp
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
phi = sàng phi tới V
for x in nums: for d in ước(x): L[d].append(x)        # giữ thứ tự gốc
ans = Σ_d phi[d] * count_increasing(L[d])
# count_increasing: Fenwick theo giá trị nén, c = 1 + prefix(rank - 1)
```

**Checklist:**

- [ ] Tăng **ngặt** (truy vấn `rank − 1`)?
- [ ] Danh sách bội giữ thứ tự gốc?
- [ ] Dùng `φ(d)` (hoặc bao hàm – loại trừ), không phải `d`?
- [ ] `% MOD` trong Fenwick?
- [ ] Đã so với vét cạn?
- [ ] Đo thời gian với mảng có nhiều ước (ví dụ toàn `55440`)?